In [ ]:
!pip install vllm transformers accelerate pandas tqdm huggingface_hub

In [ ]:
import re
import os
import json
import torch
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams

torch.set_grad_enabled(False)

In [ ]:
# Login (Run on command prompt and paste your HF token) huggingface-cli login
# Token: https://huggingface.co/settings/tokens → New token
# Request access each model you are using
# Browser: https://huggingface.co/google/gemma-3-27b-it → "Request Access"

# In Jupyter cell - NO terminal needed
from huggingface_hub import login
login(token="hf_xxxxxxxxxxxx")  # You can find token in Documents/NPS Backup folder

# Test gated model
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
print("Gated access granted!")

In [ ]:
SYSTEM_PROMPT = """
You are an expert in customer experience analysis, Net Promoter Score (NPS),
sentiment analysis, and cross-cultural interpretation of reviews.

Your task:
1. Infer sentiment on a scale from 1 to 5
2. Classify the customer as:
   - Promoter
   - Passive
   - Detractor

Important:
- Cultural norms influence how customers express satisfaction.
- Customers from high Power Distance or high Uncertainty Avoidance cultures
  may express dissatisfaction indirectly.
- Customers from indulgent cultures may exaggerate positivity.

Examples:
- Japan (high UAI): neutral wording may indicate dissatisfaction.
- USA (high IVR): positive language is more explicit.
- Germany (high UAI): criticism is direct and precise.

Use this awareness while classifying.
Return ONLY valid JSON.
"""

In [ ]:
def build_user_prompt(review_body, language, country, product_category):
    return f"""
The following is a customer review written in {language}.
The customer is from {country}.

Cultural context:
- Customers from different cultures express satisfaction and dissatisfaction differently.
- Some cultures are more indirect, cautious, or critical in wording.
- Cultural norms may affect how positivity or negativity is expressed.

Review Text:
\"\"\"{review_body}\"\"\"

Product Category:
{product_category}

Tasks:
1. Infer the sentiment of the review on a scale from 1 (very negative) to 5 (very positive).
2. Classify the customer into one NPS category:
   - Promoter
   - Passive
   - Detractor

Guidelines:
- Use the review text as the primary signal.
- Use language, country, and product category as contextual modifiers.
- Adjust interpretation based on cultural and linguistic norms.
- Be conservative when sentiment is ambiguous.
- Do not assume star-rating equivalence across cultures.

Output format (JSON only):
{{
  "sentiment_score": <1-5 integer>,
  "nps_category": "<Promoter | Passive | Detractor>"
}}
"""

In [ ]:
MODEL_NAME = "google/gemma-3-27b-it"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,        # single H100
    dtype="bfloat16",
    gpu_memory_utilization=0.90,   # safe margin
    max_model_len=4096
)

sampling_params = SamplingParams(
    temperature=0.0,
    top_p=1.0,
    max_tokens=128
)

In [ ]:

def safe_json_parse(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("No JSON found")
        return json.loads(match.group())
    except Exception:
        return {"sentiment_score": None, "nps_category": "UNKNOWN"}

def run_llm_batch(rows):
    prompts = []

    for _, row in rows.iterrows():
        user_prompt = build_user_prompt(
            review_body=row["review_body"],
            language=row["language"],
            country=row["country"],
            product_category=row["product_category"]
        )

        full_prompt = f"""<system>
                        {SYSTEM_PROMPT}
                        </system>

                        <user>
                        {user_prompt}
                        </user>
                        """
        prompts.append(full_prompt)

    outputs = llm.generate(prompts, sampling_params)

    results = []
    for out in outputs:
        text = out.outputs[0].text.strip()
        try:
            parsed = safe_json_parse(text)
            # Ensure keys exist
            parsed["sentiment_score"] = parsed.get("sentiment_score", None)
            parsed["nps_category"] = parsed.get("nps_category", "UNKNOWN")
        except Exception:
            parsed = {"sentiment_score": None, "nps_category": "UNKNOWN"}
        results.append(parsed)

    return results

In [ ]:
INPUT_CSV = "nps_raw.csv"
OUTPUT_CSV = "nps_raw_with_gemma27b.csv"

BATCH_SIZE = 32
SAVE_EVERY = 500

df = pd.read_csv(INPUT_CSV)

if "sentiment_score_gemma27b" not in df.columns:
    df["sentiment_score_gemma27b"] = None
    df["nps_category_gemma27b"] = None

na_indices = df.index[df["sentiment_score_gemma27b"].isna()]
start_idx = na_indices[0] if len(na_indices) > 0 else len(df)

print(f"Starting inference from row {start_idx}")

for i in tqdm(range(start_idx, len(df), BATCH_SIZE)):
    batch = df.iloc[i:i+BATCH_SIZE]

    results = run_llm_batch(batch)

    for idx, res in zip(batch.index, results):
        df.at[idx, "sentiment_score_gemma27b"] = res["sentiment_score"]
        df.at[idx, "nps_category_gemma27b"] = res["nps_category"]

    if i % SAVE_EVERY == 0:
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"Checkpoint saved at row {i}")

df.to_csv(OUTPUT_CSV, index=False)
print("✅ Gemma-27B inference completed")